In [1]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, f1_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
print("=" * 60)
print("CRYPTO DEEP LEARNING MODEL — TRADE CLASSIFICATION")
print("=" * 60)

df = pd.read_csv('historical_data.csv')
print(f"\n✅ Loaded {len(df):,} trades across {df['Coin'].nunique()} coins")
print(f"   Date range: {df['Timestamp IST'].min()} → {df['Timestamp IST'].max()}")

# Parse timestamp
df['Timestamp IST'] = pd.to_datetime(df['Timestamp IST'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Timestamp IST'])
df = df.sort_values('Timestamp IST').reset_index(drop=True)

CRYPTO DEEP LEARNING MODEL — TRADE CLASSIFICATION

✅ Loaded 211,224 trades across 246 coins
   Date range: 01-01-2024 01:23 → 31-12-2024 23:33


In [4]:
print("\n📐 Engineering features...")

# Time features
df['hour']        = df['Timestamp IST'].dt.hour
df['day_of_week'] = df['Timestamp IST'].dt.dayofweek
df['day_of_month']= df['Timestamp IST'].dt.day
df['month']       = df['Timestamp IST'].dt.month

# Price-based features
df['price_x_size']   = df['Execution Price'] * df['Size Tokens']
df['fee_ratio']      = df['Fee'] / (df['Size USD'] + 1e-9)
df['position_ratio'] = df['Start Position'] / (df['Size Tokens'] + 1e-9)
df['log_size_usd']   = np.log1p(df['Size USD'])
df['log_price']      = np.log1p(df['Execution Price'])
df['log_start_pos']  = np.log1p(np.abs(df['Start Position']))

# Encode categorical
le_coin = LabelEncoder()
le_side = LabelEncoder()
le_crossed = LabelEncoder()
df['coin_enc']    = le_coin.fit_transform(df['Coin'])
df['side_enc']    = le_side.fit_transform(df['Side'])
df['crossed_enc'] = le_crossed.fit_transform(df['Crossed'].astype(str))


📐 Engineering features...


In [10]:
# TASK A: Predict Side (BUY=0, SELL=1)  — binary
y_side = (df['Side'] == 'SELL').astype(int).values

# TASK B: Predict if trade is profitable (Closed PnL > 0)  — binary
# Only meaningful for trades where PnL is recorded (non-zero trades)
df_pnl = df[df['Closed PnL'] != 0].copy()
y_pnl  = (df_pnl['Closed PnL'] > 0).astype(int).values
print(f"   PnL task subset: {len(df_pnl):,} trades with non-zero PnL")
print(f"   Profitable trades: {y_pnl.mean()*100:.1f}%")
print(f"   Loss trades:       {(1-y_pnl.mean())*100:.1f}%")

FEATURE_COLS = [
    'coin_enc', 'crossed_enc',
    'Execution Price', 'Size Tokens', 'Size USD',
    'Start Position', 'Fee',
    'hour', 'day_of_week', 'day_of_month', 'month',
    'price_x_size', 'fee_ratio', 'position_ratio',
    'log_size_usd', 'log_price', 'log_start_pos'
]

X_all  = df[FEATURE_COLS].fillna(0).values
X_pnl  = df_pnl[FEATURE_COLS].fillna(0).values

print(f"\n✅ Feature matrix shape: {X_all.shape}")
print(f"   Features used: {FEATURE_COLS}")

# ─────────────────────────────────────────────────────────────
# 4. PYTORCH DATASET
# ─────────────────────────────────────────────────────────────

class TradeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

   PnL task subset: 104,408 trades with non-zero PnL
   Profitable trades: 83.2%
   Loss trades:       16.8%

✅ Feature matrix shape: (211224, 17)
   Features used: ['coin_enc', 'crossed_enc', 'Execution Price', 'Size Tokens', 'Size USD', 'Start Position', 'Fee', 'hour', 'day_of_week', 'day_of_month', 'month', 'price_x_size', 'fee_ratio', 'position_ratio', 'log_size_usd', 'log_price', 'log_start_pos']


In [6]:
class DeepTradeNet(nn.Module):
    """
    Deep Feedforward Neural Network with:
    - Batch Normalization
    - Dropout regularization
    - Residual connections
    - Sigmoid output for binary classification
    """
    def __init__(self, input_dim, hidden_dims=[256, 128, 64, 32], dropout=0.3):
        super().__init__()

        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev_dim = h

        self.network = nn.Sequential(*layers)
        self.output  = nn.Linear(prev_dim, 1)

    def forward(self, x):
        out = self.network(x)
        return self.output(out).squeeze(1)


class LSTMTradeNet(nn.Module):
    """
    LSTM-based model treating each feature as a time step
    (sequence modeling approach)
    """
    def __init__(self, input_dim, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (batch, features) → (batch, features, 1) as sequence
        x = x.unsqueeze(2)
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # last time step
        return self.fc(out).squeeze(1)



In [7]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3):
    device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model      = model.to(device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion  = nn.BCEWithLogitsLoss()

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    best_state = None

    for epoch in range(epochs):
        # ── Train
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(y_batch)

        # ── Validate
        model.eval()
        val_loss = 0; correct = 0; total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                val_loss += criterion(pred, y_batch).item() * len(y_batch)
                preds_bin = (torch.sigmoid(pred) > 0.5).float()
                correct  += (preds_bin == y_batch).sum().item()
                total    += len(y_batch)

        t_loss   = total_loss / len(train_loader.dataset)
        v_loss   = val_loss   / len(val_loader.dataset)
        v_acc    = correct / total

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        scheduler.step(v_loss)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs}  "
                  f"train_loss={t_loss:.4f}  val_loss={v_loss:.4f}  val_acc={v_acc:.4f}")

    model.load_state_dict(best_state)
    return model, history


In [8]:
def evaluate_model(model, loader, task_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits  = model(X_batch)
            probs   = torch.sigmoid(logits).cpu().numpy()
            preds   = (probs > 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy().astype(int))

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5

    print(f"\n📊 {task_name} — Test Results")
    print(f"   Accuracy : {acc:.4f} ({acc*100:.2f}%)")
    print(f"   F1 Score : {f1:.4f}")
    print(f"   ROC-AUC  : {auc:.4f}")
    print("\n" + classification_report(all_labels, all_preds,
          target_names=['Class 0', 'Class 1']))

    return all_labels, all_preds, all_probs, acc, f1, auc


In [12]:
print("\n" + "="*60)
print("TASK A: Predict Trade Side (BUY=0 vs SELL=1)")
print("="*60)

scaler_a = StandardScaler()
X_scaled_a = scaler_a.fit_transform(X_all)

X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
    X_scaled_a, y_side, test_size=0.2, random_state=42, stratify=y_side)
X_tr_a, X_va_a, y_tr_a, y_va_a = train_test_split(
    X_tr_a, y_tr_a, test_size=0.125, random_state=42, stratify=y_tr_a)

BATCH = 1024
ds_tr_a = TradeDataset(X_tr_a, y_tr_a)
ds_va_a = TradeDataset(X_va_a, y_va_a)
ds_te_a = TradeDataset(X_te_a, y_te_a)

dl_tr_a = DataLoader(ds_tr_a, batch_size=BATCH, shuffle=True,  num_workers=0)
dl_va_a = DataLoader(ds_va_a, batch_size=BATCH, shuffle=False, num_workers=0)
dl_te_a = DataLoader(ds_te_a, batch_size=BATCH, shuffle=False, num_workers=0)

print(f"\n  Train: {len(ds_tr_a):,}  Val: {len(ds_va_a):,}  Test: {len(ds_te_a):,}")
print(f"  Class balance — BUY: {(1-y_side.mean())*100:.1f}%  SELL: {y_side.mean()*100:.1f}%")

# Model A: Deep FeedForward
print("\n🧠 Training DeepTradeNet (Task A)...")
model_a = DeepTradeNet(input_dim=X_scaled_a.shape[1])
model_a, hist_a = train_model(model_a, dl_tr_a, dl_va_a, epochs=50)
labels_a, preds_a, probs_a, acc_a, f1_a, auc_a = evaluate_model(
    model_a, dl_te_a, "Task A — Side (BUY/SELL)")


TASK A: Predict Trade Side (BUY=0 vs SELL=1)

  Train: 147,856  Val: 21,123  Test: 42,245
  Class balance — BUY: 48.6%  SELL: 51.4%

🧠 Training DeepTradeNet (Task A)...
  Epoch   1/50  train_loss=0.6817  val_loss=0.6555  val_acc=0.5947
  Epoch   5/50  train_loss=0.6155  val_loss=0.5890  val_acc=0.6568
  Epoch  10/50  train_loss=0.5822  val_loss=0.5494  val_acc=0.6833
  Epoch  15/50  train_loss=0.5606  val_loss=0.5252  val_acc=0.6984
  Epoch  20/50  train_loss=0.5209  val_loss=0.5248  val_acc=0.7017
  Epoch  25/50  train_loss=0.4938  val_loss=0.5417  val_acc=0.7026
  Epoch  30/50  train_loss=0.4809  val_loss=0.5277  val_acc=0.7136
  Epoch  35/50  train_loss=0.4756  val_loss=0.5841  val_acc=0.7016
  Epoch  40/50  train_loss=0.4744  val_loss=0.5512  val_acc=0.7096
  Epoch  45/50  train_loss=0.4724  val_loss=0.5356  val_acc=0.7136
  Epoch  50/50  train_loss=0.4722  val_loss=0.4750  val_acc=0.7338

📊 Task A — Side (BUY/SELL) — Test Results
   Accuracy : 0.7386 (73.86%)
   F1 Score : 0.7384

In [15]:
print("\n" + "="*60)
print("TASK B: Predict Profitability (Loss=0 vs Profit=1)")
print("="*60)

scaler_b = StandardScaler()
X_scaled_b = scaler_b.fit_transform(X_pnl)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_scaled_b, y_pnl, test_size=0.2, random_state=42, stratify=y_pnl)
X_tr_b, X_va_b, y_tr_b, y_va_b = train_test_split(
    X_tr_b, y_tr_b, test_size=0.125, random_state=42, stratify=y_tr_b)

ds_tr_b = TradeDataset(X_tr_b, y_tr_b)
ds_va_b = TradeDataset(X_va_b, y_va_b)
ds_te_b = TradeDataset(X_te_b, y_te_b)

dl_tr_b = DataLoader(ds_tr_b, batch_size=BATCH, shuffle=True,  num_workers=0)
dl_va_b = DataLoader(ds_va_b, batch_size=BATCH, shuffle=False, num_workers=0)
dl_te_b = DataLoader(ds_te_b, batch_size=BATCH, shuffle=False, num_workers=0)

print(f"\n  Train: {len(ds_tr_b):,}  Val: {len(ds_va_b):,}  Test: {len(ds_te_b):,}")
print(f"  Class balance — Loss: {(1-y_pnl.mean())*100:.1f}%  Profit: {y_pnl.mean()*100:.1f}%")


TASK B: Predict Profitability (Loss=0 vs Profit=1)

  Train: 73,085  Val: 10,441  Test: 20,882
  Class balance — Loss: 16.8%  Profit: 83.2%


In [16]:
# Model B1: Deep FeedForward
print("\n🧠 Training DeepTradeNet (Task B)...")
model_b1 = DeepTradeNet(input_dim=X_scaled_b.shape[1])
model_b1, hist_b1 = train_model(model_b1, dl_tr_b, dl_va_b, epochs=50)
labels_b, preds_b1, probs_b1, acc_b1, f1_b1, auc_b1 = evaluate_model(
    model_b1, dl_te_b, "Task B — DNN Profitability")


🧠 Training DeepTradeNet (Task B)...
  Epoch   1/50  train_loss=0.5557  val_loss=0.4613  val_acc=0.8376
  Epoch   5/50  train_loss=0.3832  val_loss=0.3433  val_acc=0.8666
  Epoch  10/50  train_loss=0.3392  val_loss=0.2943  val_acc=0.8854
  Epoch  15/50  train_loss=0.3157  val_loss=0.2679  val_acc=0.8960
  Epoch  20/50  train_loss=0.2920  val_loss=0.2452  val_acc=0.9064
  Epoch  25/50  train_loss=0.2750  val_loss=0.2250  val_acc=0.9151
  Epoch  30/50  train_loss=0.2601  val_loss=0.2129  val_acc=0.9185
  Epoch  35/50  train_loss=0.2492  val_loss=0.1995  val_acc=0.9238
  Epoch  40/50  train_loss=0.2393  val_loss=0.1906  val_acc=0.9259
  Epoch  45/50  train_loss=0.2325  val_loss=0.1853  val_acc=0.9260
  Epoch  50/50  train_loss=0.2264  val_loss=0.1793  val_acc=0.9299

📊 Task B — DNN Profitability — Test Results
   Accuracy : 0.9317 (93.17%)
   F1 Score : 0.9271
   ROC-AUC  : 0.9611

              precision    recall  f1-score   support

     Class 0       0.91      0.66      0.76      3508

In [17]:
# Model B2: LSTM
print("\n🧠 Training LSTMTradeNet (Task B)...")
model_b2 = LSTMTradeNet(input_dim=X_scaled_b.shape[1])
model_b2, hist_b2 = train_model(model_b2, dl_tr_b, dl_va_b, epochs=50)
labels_b2, preds_b2, probs_b2, acc_b2, f1_b2, auc_b2 = evaluate_model(
    model_b2, dl_te_b, "Task B — LSTM Profitability")


🧠 Training LSTMTradeNet (Task B)...
  Epoch   1/50  train_loss=0.4879  val_loss=0.4527  val_acc=0.8320
  Epoch   5/50  train_loss=0.4555  val_loss=0.4530  val_acc=0.8320
  Epoch  10/50  train_loss=0.4555  val_loss=0.4527  val_acc=0.8320
  Epoch  15/50  train_loss=0.4554  val_loss=0.4527  val_acc=0.8320
  Epoch  20/50  train_loss=0.4555  val_loss=0.4527  val_acc=0.8320
  Epoch  25/50  train_loss=0.4553  val_loss=0.4525  val_acc=0.8320
  Epoch  30/50  train_loss=0.4551  val_loss=0.4525  val_acc=0.8320
  Epoch  35/50  train_loss=0.4546  val_loss=0.4525  val_acc=0.8320
  Epoch  40/50  train_loss=0.4552  val_loss=0.4525  val_acc=0.8320
  Epoch  45/50  train_loss=0.4546  val_loss=0.4525  val_acc=0.8320
  Epoch  50/50  train_loss=0.4544  val_loss=0.4525  val_acc=0.8320

📊 Task B — LSTM Profitability — Test Results
   Accuracy : 0.8320 (83.20%)
   F1 Score : 0.7557
   ROC-AUC  : 0.5265

              precision    recall  f1-score   support

     Class 0       0.00      0.00      0.00      350

In [20]:
print("\n📈 Generating plots...")

import os
output_dir = '/mnt/user-data/outputs/'
os.makedirs(output_dir, exist_ok=True)

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Crypto Deep Learning — Model Training Report", fontsize=16, fontweight='bold')

# ── Plot 1: Task A Loss curve
ax = axes[0, 0]
ax.plot(hist_a['train_loss'], label='Train Loss', color='royalblue')
ax.plot(hist_a['val_loss'],   label='Val Loss',   color='darkorange')
ax.set_title('Task A (Side) — Loss Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.legend(); ax.grid(alpha=0.3)

# ── Plot 2: Task A Accuracy
ax = axes[0, 1]
ax.plot(hist_a['val_acc'], color='seagreen', linewidth=2)
ax.set_title('Task A (Side) — Val Accuracy')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.axhline(max(hist_a['val_acc']), color='red', linestyle='--', alpha=0.5,
           label=f"Best: {max(hist_a['val_acc']):.4f}")
ax.legend(); ax.grid(alpha=0.3)

# ── Plot 3: Task A Confusion Matrix
ax = axes[0, 2]
cm_a = confusion_matrix(labels_a, preds_a)
sns.heatmap(cm_a, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['BUY', 'SELL'], yticklabels=['BUY', 'SELL'])
ax.set_title(f'Task A — Confusion Matrix\nAcc={acc_a:.3f}  AUC={auc_a:.3f}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# ── Plot 4: Task B (DNN) Loss curve
ax = axes[1, 0]
ax.plot(hist_b1['train_loss'], label='Train Loss', color='royalblue')
ax.plot(hist_b1['val_loss'],   label='Val Loss',   color='darkorange')
ax.set_title('Task B (DNN) — Loss Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.legend(); ax.grid(alpha=0.3)

# ── Plot 5: Task B (LSTM) Loss curve
ax = axes[1, 1]
ax.plot(hist_b2['train_loss'], label='Train Loss', color='purple')
ax.plot(hist_b2['val_loss'],   label='Val Loss',   color='crimson')
ax.set_title('Task B (LSTM) — Loss Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.legend(); ax.grid(alpha=0.3)

# ── Plot 6: Task B Model Comparison
ax = axes[1, 2]
models_b = ['DNN\n(DeepTradeNet)', 'LSTM\n(LSTMTradeNet)']
accs_b   = [acc_b1, acc_b2]
f1s_b    = [f1_b1,  f1_b2]
aucs_b   = [auc_b1, auc_b2]
x = np.arange(2)
w = 0.25
ax.bar(x - w, accs_b, w, label='Accuracy', color='royalblue')
ax.bar(x,     f1s_b,  w, label='F1',       color='seagreen')
ax.bar(x + w, aucs_b, w, label='ROC-AUC',  color='darkorange')
ax.set_title('Task B — Model Comparison')
ax.set_xticks(x); ax.set_xticklabels(models_b)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=7)

# ── Plot 7: Confusion Matrix Task B DNN
ax = axes[2, 0]
cm_b1 = confusion_matrix(labels_b, preds_b1)
sns.heatmap(cm_b1, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Loss', 'Profit'], yticklabels=['Loss', 'Profit'])
ax.set_title(f'Task B DNN — Confusion Matrix\nAcc={acc_b1:.3f}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# ── Plot 8: Confusion Matrix Task B LSTM
ax = axes[2, 1]
cm_b2 = confusion_matrix(labels_b2, preds_b2)
sns.heatmap(cm_b2, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Loss', 'Profit'], yticklabels=['Loss', 'Profit'])
ax.set_title(f'Task B LSTM — Confusion Matrix\nAcc={acc_b2:.3f}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# ── Plot 9: Dataset Summary
ax = axes[2, 2]
coin_counts = pd.read_csv('historical_data.csv')['Coin'].value_counts().head(8)
bars = ax.bar(coin_counts.index, coin_counts.values,
              color=plt.cm.tab10(np.linspace(0, 1, len(coin_counts))))
ax.set_title('Dataset — Top 8 Coins by Trade Count')
ax.set_xlabel('Coin'); ax.set_ylabel('# Trades')
ax.tick_params(axis='x', rotation=30); ax.grid(axis='y', alpha=0.3)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{int(bar.get_height()):,}', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'crypto_dl_results.png'), dpi=150, bbox_inches='tight')
print("✅ Plot saved: crypto_dl_results.png")


📈 Generating plots...
✅ Plot saved: crypto_dl_results.png


In [22]:
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

results = pd.DataFrame([
    {
        'Task': 'A — Side (BUY/SELL)',
        'Model': 'DeepTradeNet',
        'Accuracy': f'{acc_a:.4f}',
        'F1 Score': f'{f1_a:.4f}',
        'ROC-AUC':  f'{auc_a:.4f}',
        'Samples': f'{len(y_side):,}'
    },
    {
        'Task': 'B — Profitability',
        'Model': 'DeepTradeNet',
        'Accuracy': f'{acc_b1:.4f}',
        'F1 Score': f'{f1_b1:.4f}',
        'ROC-AUC':  f'{auc_b1:.4f}',
        'Samples': f'{len(y_pnl):,}'
    },
    {
        'Task': 'B — Profitability',
        'Model': 'LSTMTradeNet',
        'Accuracy': f'{acc_b2:.4f}',
        'F1 Score': f'{f1_b2:.4f}',
        'ROC-AUC':  f'{auc_b2:.4f}',
        'Samples': f'{len(y_pnl):,}'
    }
])

print(results.to_string(index=False))

# Save summary
results.to_csv('model_results_summary.csv', index=False)
print("\n✅ Summary saved: model_results_summary.csv")

# Save models
torch.save(model_a.state_dict(),  '/mnt/user-data/outputs/model_side_dnn.pt')
torch.save(model_b1.state_dict(), '/mnt/user-data/outputs/model_pnl_dnn.pt')
torch.save(model_b2.state_dict(), '/mnt/user-data/outputs/model_pnl_lstm.pt')
print("✅ Model weights saved (.pt files)")

print("\n🏁 Done!")





FINAL RESULTS SUMMARY
               Task        Model Accuracy F1 Score ROC-AUC Samples
A — Side (BUY/SELL) DeepTradeNet   0.7386   0.7384  0.8440 211,224
  B — Profitability DeepTradeNet   0.9317   0.9271  0.9611 104,408
  B — Profitability LSTMTradeNet   0.8320   0.7557  0.5265 104,408

✅ Summary saved: model_results_summary.csv
✅ Model weights saved (.pt files)

🏁 Done!
